# Building a Multi-Agent Research Pipeline with CrewAI

## What you'll learn

By the end of this notebook you will be able to:

1. Understand what **multi-agent AI systems** are and why they are useful
2. Build a custom **tool** that lets agents search the live web
3. Configure an **LLM** for use inside CrewAI
4. Define **agents** with distinct roles and goals
5. Wire **tasks** together into a sequential pipeline
6. Assemble and run a **Crew** that executes the full pipeline autonomously

---

## Background: What is CrewAI?

[CrewAI](https://github.com/joaomdmoura/crewAI) is a Python framework for orchestrating **role-playing, autonomous AI agents**. Instead of asking a single LLM to do everything, you assign each agent a specific role, equip it with relevant tools, and let the agents collaborate on tasks — much like a team of human specialists.

### Key concepts at a glance

| Concept | What it is |
|---|---|
| **Agent** | An LLM with a role, a goal, and optionally a set of tools |
| **Task** | A unit of work assigned to an agent, with a clear expected output |
| **Tool** | A Python function (or API call) the agent can invoke to gather information |
| **Crew** | The orchestrator that runs agents through their tasks in order |

---

## Scenario

A product manager at a digital knowledge platform needs to handle a rising volume of user queries that require real-time information. We will build a two-agent crew that:

1. **Researches** the latest AI advancements in interior design by searching the live web
2. **Writes** a concise executive summary from those research notes

This mirrors the classic *researcher → writer* pattern that appears across countless real-world automation use cases.

## Step 1 — Install dependencies

Before writing any agent code we need a few libraries:

| Package | Purpose |
|---|---|
| `langchain`, `langchain-openai`, `langchain-community` | LangChain core + OpenAI & community integrations used internally by CrewAI |
| `langchain-tavily`, `tavily-python` | Bindings for the [Tavily](https://tavily.com) web-search API |
| `crewai` | The main framework: Agent, Task, Crew, and Tool abstractions |
| `litellm` | A unified interface to 100+ LLM providers; CrewAI uses it under the hood |
| `python-dotenv` | Loads secrets from a `.env` file so API keys never appear in the notebook |

> **Before running:** create a `.env` file in this directory containing:
> ```
> OPENAI_API_KEY=sk-...
> OPENAI_MODEL_NAME=gpt-5-mini
> TAVILY_API_KEY=tvly-...
> ```
> You can get a free Tavily key at [app.tavily.com](https://app.tavily.com).

In [ ]:
%pip install langchain langchain-openai langchain-community 
%pip install langchain-tavily tavily-python 
%pip install crewai
%pip install litellm
%pip pydantic dotenv

## Step 2 — Import dependencies

We pull in everything we need up front:

- **`os`** — read environment variables at runtime
- **`dotenv.load_dotenv`** — parse the `.env` file and inject its values into `os.environ`
- **`requests`** — make the raw HTTP call to the Tavily search API
- **`crewai.llm.LLM`** — CrewAI's LLM wrapper (backed by LiteLLM, so it works with OpenAI, Anthropic, Mistral, and more)
- **`crewai.Agent`, `Task`, `Crew`** — the three core building blocks
- **`crewai.tools.BaseTool`** — the base class we subclass to create a custom tool

In [1]:
import os
from dotenv import load_dotenv
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

load_dotenv()  # Load environment variables from .env file

True

## Step 3 — Build a web-search tool

Agents are powerful, but their training data has a knowledge cut-off. For real-time information we give the agent a **tool** — a callable the agent can invoke whenever it decides it needs external data.

### How CrewAI tools work

CrewAI tools follow a simple contract: subclass `BaseTool`, set a `name` and `description` (the LLM reads these to decide *when* to use the tool), and implement `_run(query)`.

```
Agent thinks → picks a tool by name → calls _run(query) → uses the result in its reasoning
```

### Why Tavily?

Tavily is a search API purpose-built for LLM agents. It returns clean, structured results (title + URL) rather than raw HTML, making it easy for the LLM to reason over without token-heavy parsing.

The `_run` method below:
1. Posts the query and your API key to `api.tavily.com/search`
2. Extracts up to 5 results
3. Returns them as a newline-separated string the agent can read

In [2]:
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }
        
        response = requests.post(url, json=payload)
        data = response.json()
    
        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)
    
search_tool = TavilySearchTool()

## Step 4 — Configure the LLM

CrewAI wraps any LLM behind its `LLM` class (which delegates to **LiteLLM** internally). This means you can swap OpenAI for Anthropic, Mistral, or a local Ollama model just by changing the `model` string — no other code changes needed.

### Key parameters

| Parameter | What it controls |
|---|---|
| `model` | The model ID (read from `OPENAI_MODEL_NAME` in `.env`). E.g. `gpt-4o-mini` |
| `api_key` | Your OpenAI key, kept out of source code via `.env` |
| `is_litellm` | Tells CrewAI to route through LiteLLM for multi-provider support |
| `temperature` | Creativity vs. consistency: `0` = deterministic, `1` = highly creative. `0.5` balances both |

> **Cost tip:** `gpt-5-mini` is a great default — fast, cheap, and smart enough for most agentic tasks. Switch to `gpt-5` if you need stronger reasoning.

In [3]:
llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0.5
)

## Step 5 — Define the agents

Each **Agent** is an LLM instance given a distinct identity and purpose. CrewAI uses the `role`, `goal`, and `backstory` fields to construct the system prompt automatically — they shape *how* the LLM thinks and what it prioritises.

### The researcher agent

- `role` — how the agent identifies itself in its own reasoning
- `goal` — the outcome it is trying to achieve
- `backstory` — context that primes the LLM's persona and expertise
- `tools = [search_tool]` — the researcher gets the web search tool; the writer does not, preventing it from going off-script

### The writer agent

The writer has no tools. It relies entirely on context passed from the researcher's task output (wired in Step 6). This separation of concerns keeps each agent focused and makes the system easier to debug.

### Why use two agents instead of one?

| Single-agent approach | Two-agent approach |
|---|---|
| One LLM does everything | Each LLM does what it's best at |
| Hard to audit which step failed | Clear boundaries → easy to trace errors |
| Prompt gets long and unfocused | Short, sharp prompts per role |

> **Try it:** After running the notebook, experiment with adding a third agent (e.g. a `Fact Checker`) that validates the writer's summary against the researcher's raw notes.

In [4]:
# -- Agent --
researcher = Agent(
    role = "AI Researcher",
    goal = "Find the latest advancements in AI for interior design using web search",
    backstory = "Expert in AI research trends who gathers real-time information from the web.",
    verbose = True,
    llm = llm,
    tools = [search_tool]
)

# -- Writer --
writer = Agent(
    role = "Technical Writer",
    goal = "Summarize research into an executive report",
    backstory = "Expert in executive summaries.",
    verbose = True,
    llm = llm
)

## Step 6 — Define the tasks

A **Task** is a concrete unit of work. It tells an agent *what* to do and *what good output looks like*.

### `task_research`

- `description` — the specific instruction the agent receives (like a user message)
- `expected_output` — the LLM uses this as a quality target; it won't stop until the output matches
- `agent = researcher` — routes this task to the researcher

### `task_write`

- `context = [task_research]` — **this is the key wiring**. The writer receives `task_research`'s output automatically as additional context before it starts. This is how you build a sequential pipeline without manually passing strings between agents.
- The tight requirements in `description` (200 words, bullet points, 3 advancements) show how constraining the output format in the task description leads to more reliable, structured results.

### Data flow

```
task_research (researcher + web search)
        ↓  output injected as context
task_write (writer)
        ↓
Final executive summary
```

> **Pattern to remember:** `context = [...]` is CrewAI's way of expressing task dependencies. Tasks listed there must complete first, and their outputs are fed forward automatically.

In [5]:
# -- Tasks --
task_research = Task(
    description = "Search the web and identify the top 3 recent advancements in AI for interior design.",
    expected_output = "Detailed notes explaining three recent AI advancements in interior design with examples",
    agent = researcher
)

task_write = Task(
    description = """
    Write a concise executive summary using the reasearch notes.

    Requirements:
    - Maximum 200 words total
    - Use bullet points
    - Focus only on the 3 key advancements
    """,
    expected_output = "A bullet-point executive summary of exactly 3 items, maximum 200 words total.",
    agent = writer,
    context = [task_research]
)

## Step 7 — Assemble and run the crew

The **Crew** is the top-level orchestrator. It takes the task list and runs them in order, routing each task to its assigned agent.

### What `verbose=True` gives you

With verbosity enabled you'll see the crew's internal monologue as it runs — the agent's thought process, which tool it decides to call, what the tool returned, and how it uses that to refine its answer. This is invaluable for debugging.

### `kickoff_async` vs `kickoff`

We use `kickoff_async` (with `await`) because Jupyter notebooks run on an event loop. In a regular Python script you would use the synchronous `crew.kickoff()` instead.

### What to look for in the output

1. **Researcher's tool calls** — you'll see it formulate a search query and receive Tavily results
2. **Researcher's final answer** — the detailed notes about three AI advancements
3. **Writer receiving context** — the writer's prompt will include the researcher's notes
4. **Final output** — a clean bullet-point executive summary, ≤200 words

### What to try next

- Change the `description` in `task_research` to a different topic (e.g. AI in healthcare) — no other code changes needed
- Add `output_file="report.md"` to `task_write` to save the summary automatically
- Swap `is_litellm=True` and change `model` to `"anthropic/claude-haiku-4-5-20251001"` to run the same crew on a different provider

In [6]:
# -- Crew --
crew = Crew(
    name = "AI Research and Report Crew",
    tasks = [task_research, task_write],
    verbose = True
)

result = await crew.kickoff_async()

print("\nFinal output:\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: AI Research and Report Crew                                                                              │
│  ID: 37eabbaa-5b60-48b5-8dfc-4f49352986a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web and identify the top 3 recent advancements in AI for interior design.                     │
│  ID: 35734e9d-b0de-4186-9343-9c5a5a1f8bcc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Search the web and identify the top 3 recent advancements in AI for interior design.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'AI interior design 2024 advancements generative design tools 2024 2025'}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'AI interior design tools 2024 virtual staging generative 3D apps VRAR 2025 news'}             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - Generative design & style synthesis: AI produces multiple layout, palette and staged-photo proposals from    │
│  prompts, photos or plans using multimodal diffusion, spatial priors and catalog retrieval; enables rapid       │
│  ideation, photoreal client mockups and shoppable concepts, but risks hallucinated layouts, size/ergonomics     │
│  errors and IP issues — near-term focus: geometry-aware constraints and CAD/BIM export.                         │
│                                                                                                                 │
│  - Text-to-3D and neural rendering: diffusion-to-3D, NeRFs and lightweight mesh/point generators create         │
│  textured, view-coherent 3D rooms and assets for real-time walkthroughs and AR/VR previews; benefits include    │
│  faster asset creation, interactive visualization and scalable variant generation, while fidelity, scene        │
│  complexity and cleanup remain challenges — near-term: hybrid refinement and direct engine/CAD export.          │
│                                                                                                                 │
│  - AI-assisted material selection & sustainability-aware recommendations: material retrieval, generative        │
│  texture synthesis and predictive LCA models surface catalog matches, visualize realistic finishes and          │
│  estimate embodied carbon/cost trade-offs; advantages are faster curated choices, realistic previews and        │
│  greener alternatives, but LCA coverage and physical verification remain limited — near-term: inventory         │
│  integration and standardized material metadata.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Write a concise executive summary using the reasearch notes.                                               │
│                                                                                                                 │
│      Requirements:                                                                                              │
│      - Maximum 200 words total                                                                                  │
│      - Use bullet points                                                                                        │
│      - Focus only on the 3 key advancements                                                                     │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: AI Research and Report Crew                                                                              │
│  ID: 37eabbaa-5b60-48b5-8dfc-4f49352986a8                                                                       │
│  Final Output: - Generative design & style synthesis: AI produces multiple layout, palette and staged-photo     │
│  proposals from prompts, photos or plans using multimodal diffusion, spatial priors and catalog retrieval;      │
│  enables rapid ideation, photoreal client mockups and shoppable concepts, but risks hallucinated layouts,       │
│  size/ergonomics errors and IP issues — near-term focus: geometry-aware constraints and CAD/BIM export.         │
│                                                                                                                 │
│  - Text-to-3D and neural rendering: diffusion-to-3D, NeRFs and lightweight mesh/point generators create         │
│  textured, view-coherent 3D rooms and assets for real-time walkthroughs and AR/VR previews; benefits include    │
│  faster asset creation, interactive visualization and scalable variant generation, while fidelity, scene        │
│  complexity and cleanup remain challenges — near-term: hybrid refinement and direct engine/CAD export.          │
│                                                                                                                 │
│  - AI-assisted material selection & sustainability-aware recommendations: material retrieval, generative        │
│  texture synthesis and predictive LCA models surface catalog matches, visualize realistic finishes and          │
│  estimate embodied carbon/cost trade-offs; advantages are faster curated choices, realistic previews and        │
│  greener alternatives, but LCA coverage and physical verification remain limited — near-term: inventory         │
│  integration and standardized material metadata.                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final output:

- Generative design & style synthesis: AI produces multiple layout, palette and staged-photo proposals from prompts, photos or plans using multimodal diffusion, spatial priors and catalog retrieval; enables rapid ideation, photoreal client mockups and shoppable concepts, but risks hallucinated layouts, size/ergonomics errors and IP issues — near-term focus: geometry-aware constraints and CAD/BIM export.

- Text-to-3D and neural rendering: diffusion-to-3D, NeRFs and lightweight mesh/point generators create textured, view-coherent 3D rooms and assets for real-time walkthroughs and AR/VR previews; benefits include faster asset creation, interactive visualization and scalable variant generation, while fidelity, scene complexity and cleanup remain challenges — near-term: hybrid refinement and direct engine/CAD export.

- AI-assisted material selection & sustainability-aware recommendations: material retrieval, generative texture synthesis and predictive LCA models surface ca

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯